# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Neel0289/FlyRank-Week1A1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

I use a Random Forest Regressor because this lane is a ranking/scoring problem rather than a simple yes/no classification task. The model can combine several SEO and engagement signals and produce a continuous opportunity score that can be used to rank pages. I chose a tree-based model because the relationships between CTR, search demand, engagement, trend, and content characteristics may be nonlinear. The goal is not to reward model complexity by itself, but to test whether combining these signals improves ranking quality over the Week-4 rule-based baseline. The target used in this notebook is an explicitly defined proxy, not an observed ground-truth opportunity label.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

I use a grouped train/test split by client_id so that pages from the same client do not appear in both training and test sets. This makes the evaluation more conservative because the model must generalize to unseen clients rather than benefiting from repeated client-specific patterns. An 80/20 grouped split is used with a fixed random seed for reproducibility. The available starter CSV does not contain an explicit date column, so a true time-based March-to-June validation cannot be implemented from this file alone.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import ndcg_score

# --------------------------------------------------
# 1. Load the same dataset used in Week 4
# --------------------------------------------------

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

# --------------------------------------------------
# 2. Feature columns
# --------------------------------------------------

feature_cols = [
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "engaged_sessions_90d",
    "search_volume",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "ai_traffic_pct",
    "content_age_days",
    "days_since_last_update",
    "trend_pct"
]

# Keep only rows with the required model inputs
model_df = df.dropna(
    subset=feature_cols + ["client_id"]
).copy()

print("Rows available for modeling:", len(model_df))
print("Clients available:", model_df["client_id"].nunique())

# --------------------------------------------------
# 3. Define proxy relevance
# --------------------------------------------------
# This is NOT ground truth.
# It is a transparent proxy made from engagement/trend signals
# that were not used by the Week-4 baseline rule.

model_df["proxy_relevance"] = (
    model_df["engagement_rate"].rank(pct=True) * 0.5
    + model_df["scroll_rate"].rank(pct=True) * 0.2
    + model_df["trend_pct"].rank(pct=True) * 0.3
)

# --------------------------------------------------
# 4. Grouped train/test split by client
# --------------------------------------------------

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        model_df,
        groups=model_df["client_id"]
    )
)

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("\nTrain rows:", len(train_df))
print("Test rows :", len(test_df))

print("\nTrain clients:", train_df["client_id"].nunique())
print("Test clients :", test_df["client_id"].nunique())

print(
    "\nClient overlap:",
    len(
        set(train_df["client_id"])
        & set(test_df["client_id"])
    )
)

Dataset shape: (30000, 44)
Rows available for modeling: 25188
Clients available: 31

Train rows: 20624
Test rows : 4564

Train clients: 24
Test clients : 7

Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

I train the Random Forest using the grouped training set and evaluate both the model and the Week-4 baseline on exactly the same held-out test pages. The evaluation metric is NDCG, matching the ranking objective defined in Week 2. Because the dataset does not provide a directly observed opportunity label, the NDCG relevance values are based on the proxy relevance definition above and should be interpreted as directional evidence rather than proof that one method predicts future business outcomes.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------
# 1. Train Random Forest
# --------------------------------------------------

X_train = train_df[feature_cols]
y_train = train_df["proxy_relevance"]

X_test = test_df[feature_cols]
y_test = test_df["proxy_relevance"]

rf = RandomForestRegressor(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

# --------------------------------------------------
# 2. Model predictions
# --------------------------------------------------

test_df["model_score"] = rf.predict(X_test)

# --------------------------------------------------
# 3. Recreate the Week-4 baseline on the TEST SET only
# --------------------------------------------------

baseline_df = test_df.dropna(
    subset=["ctr", "avg_position", "search_volume"]
).copy()

# Position buckets exactly matching Week 4
position_bins = [0, 1, 3, 5, 10, 20, np.inf]

position_labels = [
    "1",
    "2-3",
    "4-5",
    "6-10",
    "11-20",
    "21+"
]

baseline_df["position_bucket"] = pd.cut(
    baseline_df["avg_position"],
    bins=position_bins,
    labels=position_labels,
    include_lowest=True
)

# Expected CTR from the TEST SET
expected_ctr = (
    baseline_df
    .groupby(
        "position_bucket",
        observed=True
    )["ctr"]
    .median()
)

baseline_df["expected_ctr"] = (
    baseline_df["position_bucket"]
    .map(expected_ctr)
)

baseline_df["ctr_gap"] = (
    baseline_df["expected_ctr"]
    - baseline_df["ctr"]
).clip(lower=0)

ctr_max = baseline_df["ctr_gap"].max()

if ctr_max > 0:
    baseline_df["ctr_gap_score"] = (
        baseline_df["ctr_gap"] / ctr_max
    )
else:
    baseline_df["ctr_gap_score"] = 0

baseline_df["volume_log"] = np.log1p(
    baseline_df["search_volume"]
)

volume_max = baseline_df["volume_log"].max()

if volume_max > 0:
    baseline_df["volume_score"] = (
        baseline_df["volume_log"] / volume_max
    )
else:
    baseline_df["volume_score"] = 0

baseline_df["baseline_score"] = (
    0.7 * baseline_df["ctr_gap_score"]
    + 0.3 * baseline_df["volume_score"]
)

# --------------------------------------------------
# 4. Evaluate both rankings using NDCG
# --------------------------------------------------

y_true = baseline_df["proxy_relevance"].to_numpy()

model_scores = baseline_df["model_score"].to_numpy()

baseline_scores = baseline_df["baseline_score"].to_numpy()

model_ndcg = ndcg_score(
    [y_true],
    [model_scores],
    k=20
)

baseline_ndcg = ndcg_score(
    [y_true],
    [baseline_scores],
    k=20
)

comparison = pd.DataFrame({
    "Method": [
        "Week-4 baseline",
        "Week-5 Random Forest"
    ],
    "Metric": [
        "NDCG@20",
        "NDCG@20"
    ],
    "Score": [
        baseline_ndcg,
        model_ndcg
    ]
})

display(comparison)

print(
    f"\nNDCG improvement:",
    round(model_ndcg - baseline_ndcg, 4)
)

,Method,Metric,Score
0,Week-4 baseline,NDCG@20,0.345621
1,Week-5 Random Forest,NDCG@20,0.998915



NDCG improvement: 0.6533


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

The error analysis focuses on pages where the model and baseline disagree strongly. I also inspect permutation importance to understand which signals the model relies on. These results are interpreted as directional evidence about the available dataset, not as proof of causal relationships. In particular, a high feature importance does not mean that changing that feature will cause an improvement in clicks or engagement.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --------------------------------------------------
# 1. Largest disagreements between model and baseline
# --------------------------------------------------

baseline_df["rank_model"] = (
    baseline_df["model_score"]
    .rank(
        ascending=False,
        method="first"
    )
)

baseline_df["rank_baseline"] = (
    baseline_df["baseline_score"]
    .rank(
        ascending=False,
        method="first"
    )
)

baseline_df["rank_difference"] = (
    baseline_df["rank_baseline"]
    - baseline_df["rank_model"]
).abs()

disagreements = (
    baseline_df
    .sort_values(
        "rank_difference",
        ascending=False
    )
    [
        [
            "content_id",
            "proxy_relevance",
            "model_score",
            "baseline_score",
            "rank_model",
            "rank_baseline",
            "rank_difference",
            "ctr",
            "avg_position",
            "search_volume",
            "engagement_rate",
            "scroll_rate",
            "trend_pct"
        ]
    ]
    .head(10)
)

print("Largest model-vs-baseline disagreements:")
display(disagreements)

# --------------------------------------------------
# 2. Permutation importance
# --------------------------------------------------

perm = permutation_importance(
    rf,
    X_test,
    y_test,
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

importance = (
    pd.DataFrame({
        "feature": feature_cols,
        "importance": perm.importances_mean
    })
    .sort_values(
        "importance",
        ascending=False
    )
)

print("\nTop model features:")
display(importance.head(10))

# --------------------------------------------------
# 3. Concise interpretation
# --------------------------------------------------

top_features = importance.head(3)["feature"].tolist()

print(
    "\nInterpretation:"
)
print(
    "The largest disagreements identify pages where the flexible model "
    "and the Week-4 heuristic prioritize different pages."
)
print(
    "The Random Forest relies most on these signals:",
    ", ".join(top_features)
)
print(
    "These are associations in the observed data, not causal effects."
)

Largest model-vs-baseline disagreements:


,content_id,proxy_relevance,model_score,baseline_score,rank_model,rank_baseline,rank_difference,ctr,avg_position,search_volume,engagement_rate,scroll_rate,trend_pct
27524,content_addafe3df89b,0.214396,0.214396,0.786958,4543.0,44.0,4499.0,0.00,5.4,2900.0,0.00,0.00,-100.0
29662,content_5c7dda288778,0.897026,0.895617,0.000000,63.0,4555.0,4492.0,1.33,7.8,0.0,15.85,12.15,82.7
21867,content_608540486d95,0.214396,0.214396,0.817140,4500.0,23.0,4477.0,0.00,7.0,8100.0,0.00,0.00,-100.0
23481,content_0a1a7fb81747,0.214396,0.214396,0.789481,4515.0,40.0,4475.0,0.00,4.8,20.0,0.00,0.00,-100.0
28299,content_2107a16ad28d,0.214396,0.214396,0.770476,4550.0,78.0,4472.0,0.00,4.8,10.0,0.00,0.00,-100.0
20807,content_ae6a3adf7533,0.214396,0.214396,0.832577,4489.0,20.0,4469.0,0.00,4.8,90.0,0.00,0.00,-100.0
26881,content_f09dd421b34e,0.214396,0.214396,0.770476,4536.0,76.0,4460.0,0.00,3.5,10.0,0.00,0.00,-100.0
17407,content_31b10f132c97,0.214396,0.214396,0.934326,4459.0,1.0,4458.0,0.00,3.3,2900.0,0.00,0.00,-100.0
16417,content_f4b3081037b3,0.214396,0.214396,0.832577,4449.0,19.0,4430.0,0.00,5.0,90.0,0.00,0.00,-100.0
25158,content_51bcdbbb0b67,0.214396,0.214396,0.713274,4530.0,114.0,4416.0,0.00,10.8,2900.0,0.00,0.00,-100.0



Top model features:


,feature,importance
7,engagement_rate,4.771139e-01
12,trend_pct,4.070943e-01
8,scroll_rate,1.805946e-01
3,engaged_sessions_90d,7.828267e-02
2,sessions_90d,1.029382e-05
0,impressions_90d,4.031296e-06
6,avg_position,5.671000e-07
5,ctr,5.346758e-07
4,search_volume,3.570238e-08
9,ai_traffic_pct,7.742885e-10



Interpretation:
The largest disagreements identify pages where the flexible model and the Week-4 heuristic prioritize different pages.
The Random Forest relies most on these signals: engagement_rate, trend_pct, scroll_rate
These are associations in the observed data, not causal effects.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.